# GRU et ODE-RNN Survival AFT

Ce notebook développe, entraîne et compare deux modèles de survie basés sur des réseaux de neurones récurrents pour prédire le temps restant avant l'insuffisance rénale terminale (IRT) chez les patients atteints de polykystose rénale autosomique dominante (PKRAD) : un modèle **GRU Survival AFT** et un modèle **ODE-RNN Survival AFT**.

Le modèle **GRU Survival AFT** exploite les trajectoires longitudinales des patients afin d'apprendre une représentation temporelle des données cliniques. Le modèle **ODE-RNN Survival AFT** étend cette approche en intégrant un bloc d'équations différentielles ordinaires (ODE) permettant de modéliser l'évolution continue de l'état latent entre deux visites successives. Dans les deux cas, une tête **Accelerated Failure Time (AFT)** est utilisée pour prédire le temps restant avant l'IRT.

Le notebook couvre l'ensemble du pipeline de modélisation :

- chargement et préparation des jeux de données longitudinaux ;
- construction des jeux de données dynamiques et des DataLoaders PyTorch ;
- définition des architectures **GRU Survival AFT** et **ODE-RNN Survival AFT** ;
- optimisation des hyperparamètres par **Optuna** avec validation croisée **StratifiedGroupKFold** au niveau des patients ;
- réentraînement des meilleurs modèles sur l'ensemble des données de développement ;
- sauvegarde et rechargement des modèles finaux ;
- prédictions sur les jeux de test interne et externe ;
- évaluation des performances à l'aide du **Harrell's C-index**, de la **MAE** et de la **RMSE**, globalement et selon le nombre de visites disponibles ;
- analyse des principales sur-estimations et sous-estimations des prédictions ;
- comparaison détaillée des performances et des prédictions des modèles **GRU Survival AFT** et **ODE-RNN Survival AFT**.

L'objectif est d'évaluer l'apport de la modélisation continue introduite par l'ODE-RNN par rapport à une architecture GRU classique pour la prédiction du temps avant l'IRT à partir de données longitudinales.

---
## 0. Imports

In [ ]:
# ==========================================================
# IMPORTATION DES BIBLIOTHÈQUES
# ==========================================================

# ==========================================================
# MANIPULATION DES DONNÉES
# ==========================================================
import polars as pl
import pandas as pd
import numpy as np
import random

# ==========================================================
# VISUALISATION
# ==========================================================
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ==========================================================
# JAX / NEURAL ODE
# ==========================================================
import jax
import jax.numpy as jnp
import jax.random as jr
import jax.scipy as jsp
import equinox as eqx
import diffrax
import optax

# ==========================================================
# PYTORCH
# ==========================================================
import torch
import torch.nn as nn
import torch.nn.functional as F

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import (
    pad_sequence,
    pack_padded_sequence
)
from torch.distributions import LogNormal

# ==========================================================
# ANALYSE DE SURVIE
# ==========================================================
from lifelines.utils import concordance_index

# ==========================================================
# MÉTRIQUES D'ÉVALUATION
# ==========================================================
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error
)

# ==========================================================
# OPTIMISATION DES HYPERPARAMÈTRES
# ==========================================================
import optuna

# ==========================================================
# STATISTIQUES
# ==========================================================
from scipy.stats import pearsonr

# ==========================================================
# UTILITAIRES
# ==========================================================
from tqdm import tqdm
import time
import copy
import pickle

# ==========================================================
# INFORMATIONS SUR L'ENVIRONNEMENT D'EXÉCUTION
# ==========================================================
print("JAX version :", jax.__version__)
print("Devices disponibles :", jax.devices())

In [ ]:
import sys
print(sys.executable)

import jax
print(jax.devices())

In [ ]:
# ==========================================================
# REPRODUCTIBILITÉ
# ==========================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)

torch.manual_seed(SEED)

if torch.cuda.is_available():

    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)


torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False


torch.use_deterministic_algorithms(False)


## 1. Chargement des données


In [ ]:
# ==========================================================
# CHARGEMENT DES JEUX DE DONNÉES
# ==========================================================
#
# Trois jeux de données longitudinaux sont utilisés :
#
# - train_dynamic :
#     utilisé pour l'entraînement du modèle, la recherche des
#     hyperparamètres et la validation croisée.
#
# - internal_test_dynamic :
#     conservé entièrement à l'écart pendant le développement.
#     Il est utilisé uniquement pour la validation interne du
#     modèle final.
#
# - external_test_dynamic :
#     cohorte indépendante utilisée uniquement pour la
#     validation externe afin d'évaluer la capacité de
#     généralisation du modèle.
# ==========================================================

DATA_DIR = "./data/My_data"

train_dynamic = pl.read_parquet(f"{DATA_DIR}/train_dynamic.parquet")
internal_test_dynamic = pl.read_parquet(f"{DATA_DIR}/internal_test_dynamic.parquet")
external_test_dynamic = pl.read_parquet(f"{DATA_DIR}/external_test_dynamic.parquet")

print("✅ Toutes les colonnes requises sont présentes.")
print(f"Jeu d'entraînement : {train_dynamic.shape}")
print(f"Jeu de validation interne : {internal_test_dynamic.shape}")
print(f"Jeu de validation externe : {external_test_dynamic.shape}")

train_dynamic

---
## 2. Vérifications exploratoires

In [ ]:
# ==========================================================
# DISTRIBUTION DU NOMBRE DE MESURES PAR PATIENT
# ==========================================================
#
# Cette cellule résume le nombre de visites longitudinales
# disponibles dans le jeu d'entraînement.
#
# La variable `n_measures` correspond au nombre total de
# mesures d'eGFR disponibles pour chaque patient.
#
# Cette distribution permet notamment de :
# - connaître la richesse du suivi longitudinal ;
# - identifier le nombre maximal de visites disponibles ;
# - anticiper la taille des différents snapshots (k = 1, 2, ..., K).
# ==========================================================

n_mesures_dist = (
    train_dynamic
    .select(pl.col("n_measures"))
    .group_by("n_measures")
    .len()
    .sort("n_measures")
)

print(n_mesures_dist)

In [ ]:
# ==========================================================
# TAUX D'ÉVÉNEMENTS DANS CHAQUE JEU DE DONNÉES
# ==========================================================
#
# Cette cellule résume la proportion de patients ayant atteint
# l'insuffisance rénale terminale (IRT) dans chacun des trois
# jeux de données :
#
# - entraînement ;
# - validation interne ;
# - validation externe.
#
# Ces statistiques permettent de vérifier que la répartition
# des événements et des censures est cohérente entre les
# différentes cohortes.
# ==========================================================

datasets = {
    "Entraînement": train_dynamic,
    "Validation interne": internal_test_dynamic,
    "Validation externe": external_test_dynamic,
}

for name, df in datasets.items():
    events = df["event_irt"].to_numpy()

    print(f"\n{name}")
    print("-" * len(name))
    print(f"Événements (IRT) : {events.sum()}")
    print(f"Censures         : {len(events) - events.sum()}")
    print(f"Taux d'événements: {events.mean():.3f}")

#### Préparation des données pour GUR-AFT et ODE-RNN

In [ ]:
# ==========================================================
# CONSTRUCTION DES SÉQUENCES LONGITUDINALES
# ==========================================================
#
# Cette étape regroupe les variables longitudinales de chaque
# patient en une séquence chronologique qui servira d'entrée
# au modèle ODE-RNN / GRU.
#
# Chaque élément de la séquence correspond à une visite et
# contient les variables suivantes :
#
#   [âge, eGFR, hypertension, événement urologique, temps]
#
# La colonne `sequence_features` est créée pour chacun des
# trois jeux de données :
#   - entraînement ;
#   - validation interne ;
#   - validation externe.
# ==========================================================

def build_sequence_features(row):
    """
    Construit la séquence longitudinale d'un patient.

    Chaque visite est représentée par :
        [âge, eGFR, hypertension, événement urologique, temps]

    Returns
    -------
    list[list[float]]
        Séquence chronologique des visites.
    """

    return [
        [age, egfr, hta, uro, time]
        for age, egfr, hta, uro, time in zip(
            row["age_history"],
            row["egfr_history"],
            row["hypertension_history"],
            row["urologic_history"],
            row["time_history"],
        )
    ]


# ==========================================================
# APPLICATION À CHAQUE JEU DE DONNÉES
# ==========================================================

datasets = {
    "train": train_dynamic,
    "internal": internal_test_dynamic,
    "external": external_test_dynamic,
}

for name, df in datasets.items():

    datasets[name] = df.with_columns(
        pl.struct(
            [
                "age_history",
                "egfr_history",
                "hypertension_history",
                "urologic_history",
                "time_history",
            ]
        )
        .map_elements(
            build_sequence_features,
            return_dtype=pl.List(pl.List(pl.Float64)),
        )
        .alias("sequence_features")
    )


# ==========================================================
# MISE À JOUR DES JEUX DE DONNÉES
# ==========================================================

train_dynamic = datasets["train"]
internal_test_dynamic = datasets["internal"]
external_test_dynamic = datasets["external"]

In [ ]:
# ==========================================================
# APERÇU DES SÉQUENCES LONGITUDINALES
# ==========================================================
#
# Vérification de la colonne `sequence_features` créée lors de
# l'étape précédente.
#
# Les informations affichées permettent de contrôler :
#   - l'identifiant du patient ;
#   - la séquence longitudinale construite ;
#   - le nombre de visites disponibles ;
#   - le temps restant avant l'événement ou la censure.
# ==========================================================

train_dynamic.select(
    [
        "patient_id",
        "sequence_features",
        "n_measures",
        "time_to_event",
    ]
).head()

#### Calcul des statistques

In [ ]:
# ==========================================================
# CALCUL DES PARAMÈTRES DE NORMALISATION
# ==========================================================
#
# Les variables continues utilisées par le modèle sont
# normalisées à l'aide de leur moyenne et de leur écart-type
# calculés exclusivement sur le jeu d'entraînement.
#
# Ces paramètres seront ensuite appliqués, sans
# recalcul, aux jeux de validation interne et externe afin
# d'éviter toute fuite d'information.
# ==========================================================

# ==========================================================
# EXTRACTION DES VARIABLES CONTINUES
# ==========================================================

all_ages = np.concatenate(
    train_dynamic["age_history"].to_list()
)

all_egfr = np.concatenate(
    train_dynamic["egfr_history"].to_list()
)

all_times = np.concatenate(
    train_dynamic["time_history"].to_list()
)

# ==========================================================
# CALCUL DES MOYENNES ET ÉCARTS-TYPES
# ==========================================================

age_mean = all_ages.mean()
age_std = all_ages.std()

egfr_mean = all_egfr.mean()
egfr_std = all_egfr.std()

time_mean = all_times.mean()
time_std = all_times.std()

# ==========================================================
# AFFICHAGE DES PARAMÈTRES DE NORMALISATION
# ==========================================================

print(f"Age  : mean = {age_mean:.2f} | std = {age_std:.2f}")
print(f"eGFR : mean = {egfr_mean:.2f} | std = {egfr_std:.2f}")
print(f"Time : mean = {time_mean:.2f} | std = {time_std:.2f}")

In [ ]:
# ==========================================================
# SAUVEGARDE DES SÉQUENCES ORIGINALES
# ==========================================================
#
# Une copie des séquences longitudinales est conservée avant
# leur normalisation. Cette sauvegarde permet :
#   - de conserver les valeurs originales pour les analyses
#     et les visualisations ;
#   - de vérifier les transformations appliquées ;
#   - de reconstruire les données si nécessaire.
#
# La colonne `sequence_features` sera ensuite remplacée par
# sa version normalisée.
# ==========================================================

datasets = {
    "train": train_dynamic,
    "internal": internal_test_dynamic,
    "external": external_test_dynamic,
}

for name, df in datasets.items():
    datasets[name] = df.with_columns(
        pl.col("sequence_features").alias("sequence_features_raw")
    )

train_dynamic = datasets["train"]
internal_test_dynamic = datasets["internal"]
external_test_dynamic = datasets["external"]

#### Normalisation de sequence_features

In [ ]:
# ==========================================================
# FONCTION DE NORMALISATION DES SÉQUENCES LONGITUDINALES
# ==========================================================
#
# Cette fonction applique une standardisation (z-score) aux
# variables continues de chaque visite d'un patient.
#
# Variables normalisées :
#   - âge
#   - eGFR
#   - temps
#
# Les variables catégorielles/binaires (hypertension et
# événement urologique) sont conservées inchangées.
#
# Les paramètres de normalisation (moyenne et écart-type)
# sont calculés uniquement sur le jeu d'entraînement puis
# réutilisés pour les jeux de validation.
# ==========================================================

def normalize_sequence_features(
    sequence,
    age_mean,
    age_std,
    egfr_mean,
    egfr_std,
    time_mean,
    time_std,
):
    """
    Normalise les variables continues d'une séquence longitudinale.

    Parameters
    ----------
    sequence : list[list[float]]
        Séquence de visites d'un patient.

    Returns
    -------
    list[list[float]]
        Séquence normalisée.
    """

    normalized_sequence = []

    for age, egfr, hta, uro, time in sequence:

        normalized_sequence.append(
            [
                (age - age_mean) / age_std,
                (egfr - egfr_mean) / egfr_std,
                hta,
                uro,
                (time - time_mean) / time_std,
            ]
        )

    return normalized_sequence

In [ ]:
# ==========================================================
# APPLICATION DE LA NORMALISATION AUX SÉQUENCES
# ==========================================================
#
# Les séquences longitudinales sont normalisées à l'aide des
# paramètres (moyenne et écart-type) estimés exclusivement sur
# le jeu d'entraînement.
#
# Les mêmes paramètres sont ensuite appliqués aux jeux de
# validation interne et externe afin de garantir une
# évaluation indépendante et d'éviter toute fuite
# d'information.
# ==========================================================

datasets = {
    "train": train_dynamic,
    "internal": internal_test_dynamic,
    "external": external_test_dynamic,
}

for name, df in datasets.items():

    datasets[name] = df.with_columns(

        pl.col("sequence_features")
        .map_elements(
            lambda seq: normalize_sequence_features(
                seq,
                age_mean,
                age_std,
                egfr_mean,
                egfr_std,
                time_mean,
                time_std,
            ),
            return_dtype=pl.List(pl.List(pl.Float64)),
        )
        .alias("sequence_features")

    )

# ==========================================================
# MISE À JOUR DES JEUX DE DONNÉES
# ==========================================================

train_dynamic = datasets["train"]
internal_test_dynamic = datasets["internal"]
external_test_dynamic = datasets["external"]

#### Encodage des variables statiques 

In [ ]:
# ==========================================================
# ENCODAGE DES VARIABLES STATIQUES
# ==========================================================
#
# Les variables statiques sont converties en représentation
# numérique afin d'être concaténées à la représentation
# latente produite par le modèle ODE-RNN / GRU.
#
# Variables encodées :
#   - sexe (one-hot)
#   - génotype (one-hot)
#
# Les mêmes transformations sont appliquées aux jeux
# d'entraînement, de validation interne et de validation
# externe afin de garantir une représentation identique des
# données.
# ==========================================================

datasets = {
    "train": train_dynamic,
    "internal": internal_test_dynamic,
    "external": external_test_dynamic,
}

# ==========================================================
# ENCODAGE ONE-HOT DU SEXE
# ==========================================================

for name, df in datasets.items():

    datasets[name] = df.with_columns(

        [
            (pl.col("sex") == "Female")
            .cast(pl.Float32)
            .alias("sex_female"),

            (pl.col("sex") == "Male")
            .cast(pl.Float32)
            .alias("sex_male"),
        ]

    )

# ==========================================================
# ENCODAGE ONE-HOT DU GÉNOTYPE
# ==========================================================

genetic_categories = [

    "other",
    "PKD2",
    "PKD1_non_truncating",
    "PKD1_truncating",
    "GUR",

]

for name, df in datasets.items():

    datasets[name] = df.with_columns(

        [
            (pl.col("genetic") == category)
            .cast(pl.Float32)
            .alias(f"genetic_{category}")

            for category in genetic_categories
        ]

    )

# ==========================================================
# VARIABLES STATIQUES UTILISÉES PAR LE MODÈLE
# ==========================================================
#
# Ces variables seront concaténées en un vecteur unique
# transmis au réseau de neurones en complément des
# informations longitudinales.
# ==========================================================

static_columns = [

    # Sexe
    "sex_female",
    "sex_male",

    # Génotype
    "genetic_other",
    "genetic_PKD2",
    "genetic_PKD1_non_truncating",
    "genetic_PKD1_truncating",
    "genetic_GUR",

]

# ==========================================================
# CONSTRUCTION DU VECTEUR DE FEATURES STATIQUES
# ==========================================================

for name, df in datasets.items():

    datasets[name] = df.with_columns(

        pl.concat_list(static_columns)
        .alias("static_features")

    )

# ==========================================================
# MISE À JOUR DES JEUX DE DONNÉES
# ==========================================================

train_dynamic = datasets["train"]
internal_test_dynamic = datasets["internal"]
external_test_dynamic = datasets["external"]

In [ ]:
# ==========================================================
# DATASET PYTORCH POUR LES DONNÉES LONGITUDINALES
# ==========================================================
#
# Cette classe convertit le DataFrame Polars en un Dataset
# PyTorch compatible avec un DataLoader.
#
# Chaque observation correspond à un snapshot longitudinal
# d'un patient et contient :
#
#   - la séquence longitudinale ;
#   - les variables statiques ;
#   - le temps jusqu'à l'événement ou la censure ;
#   - l'indicateur d'événement ;
#   - la longueur réelle de la séquence ;
#   - l'identifiant du patient.
#
# Les séquences peuvent avoir des longueurs différentes.
# Elles seront homogénéisées ultérieurement par la fonction
# `collate_fn`.
# ==========================================================

class DynamicSurvivalDataset(Dataset):
    """
    Dataset PyTorch pour la prédiction de survie à partir
    de données longitudinales.
    """

    def __init__(self, df):
        """
        Parameters
        ----------
        df : polars.DataFrame
            Jeu de données dynamique.
        """
        self.df = df

    # --------------------------------------------------
    # Nombre total de snapshots
    # --------------------------------------------------

    def __len__(self):
        return self.df.height

    # --------------------------------------------------
    # Extraction d'un snapshot
    # --------------------------------------------------

    def __getitem__(self, idx):

        row = self.df.row(idx, named=True)

        # --------------------------------------------------
        # Séquence longitudinale
        #
        # Shape :
        #   (longueur_de_séquence, 5)
        #
        # Variables :
        #   [âge, eGFR, hypertension,
        #    événement urologique, temps]
        # --------------------------------------------------

        sequence = torch.tensor(
            row["sequence_features"],
            dtype=torch.float32,
        )

        # --------------------------------------------------
        # Variables statiques
        #
        # Shape :
        #   (n_static_features,)
        # --------------------------------------------------

        static = torch.tensor(
            row["static_features"],
            dtype=torch.float32,
        )

        # --------------------------------------------------
        # Temps jusqu'à l'événement ou la censure
        # --------------------------------------------------

        time_to_event = torch.tensor(
            row["time_to_event"],
            dtype=torch.float32,
        )

        # --------------------------------------------------
        # Indicateur d'événement
        #
        # 1 = IRT observée
        # 0 = patient censuré
        # --------------------------------------------------

        event = torch.tensor(
            row["event_irt"],
            dtype=torch.float32,
        )

        # --------------------------------------------------
        # Longueur réelle de la séquence
        # --------------------------------------------------

        length = sequence.shape[0]

        return {
            "sequence": sequence,
            "static": static,
            "time_to_event": time_to_event,
            "event": event,
            "length": length,
            "patient_id": row["patient_id"],
        }

In [ ]:
# ==========================================================
# COLLATE FUNCTION POUR LES DONNÉES LONGITUDINALES
# ==========================================================
#
# Cette fonction est utilisée par le DataLoader pour regrouper
# plusieurs snapshots en un mini-batch.
#
# Les séquences longitudinales pouvant avoir des longueurs
# différentes, elles sont complétées (padding) avec des zéros
# jusqu'à la longueur maximale du batch.
#
# La longueur réelle de chaque séquence est conservée afin que
# le modèle puisse ignorer les valeurs de padding lors du
# traitement.
# ==========================================================

def survival_collate_fn(batch):
    """
    Construit un mini-batch PyTorch à partir d'une liste
    d'observations.

    Parameters
    ----------
    batch : list[dict]
        Liste d'observations retournées par
        DynamicSurvivalDataset.

    Returns
    -------
    dict
        Batch contenant les séquences paddées, les variables
        statiques, les temps jusqu'à l'événement, les
        indicateurs d'événement, les longueurs réelles des
        séquences et les identifiants des patients.
    """

    # --------------------------------------------------
    # Séquences longitudinales
    # --------------------------------------------------

    sequences = [
        item["sequence"]
        for item in batch
    ]

    # --------------------------------------------------
    # Padding des séquences
    #
    # Shape :
    #   (batch_size, longueur_max, 5)
    # --------------------------------------------------

    padded_sequences = pad_sequence(
        sequences,
        batch_first=True,
        padding_value=0.0,
    )

    # --------------------------------------------------
    # Longueurs réelles des séquences
    # --------------------------------------------------

    lengths = torch.tensor(
        [item["length"] for item in batch],
        dtype=torch.long,
    )

    # --------------------------------------------------
    # Variables statiques
    # --------------------------------------------------

    static = torch.stack(
        [item["static"] for item in batch]
    )

    # --------------------------------------------------
    # Temps jusqu'à l'événement ou la censure
    # --------------------------------------------------

    time_to_event = torch.stack(
        [item["time_to_event"] for item in batch]
    )

    # --------------------------------------------------
    # Indicateur d'événement
    # --------------------------------------------------

    event = torch.stack(
        [item["event"] for item in batch]
    )

    # --------------------------------------------------
    # Identifiants des patients
    # --------------------------------------------------

    patient_ids = [
        item["patient_id"]
        for item in batch
    ]

    return {

        "sequence": padded_sequences,
        "lengths": lengths,
        "static": static,
        "time_to_event": time_to_event,
        "event": event,
        "patient_id": patient_ids,

    }

In [ ]:
# ==========================================================
# MODÈLE GRU - LOGNORMAL AFT
# ==========================================================
#
# Le modèle est composé de deux parties :
#
#   1. Un encodeur GRU qui résume la séquence
#      longitudinale de chaque patient.
#
#   2. Une tête LogNormal AFT qui combine la représentation
#      longitudinale avec les variables statiques afin de
#      prédire les paramètres (μ, σ) de la distribution
#      Log-Normale du temps jusqu'à l'événement.
#
# Les séquences de longueurs variables sont gérées grâce au
# mécanisme de packing de PyTorch.
# ==========================================================

class GRUAFT(nn.Module):
    """
    Modèle GRU pour la prédiction du temps jusqu'à l'IRT
    avec une sortie LogNormal AFT.
    """

    def __init__(
        self,
        input_dim=5,
        static_dim=7,
        hidden_dim=64,
        dropout=0.0,
    ):
        super().__init__()

        # --------------------------------------------------
        # Encodeur longitudinal
        # --------------------------------------------------

        self.gru = nn.GRU(
            input_size=input_dim,
            hidden_size=hidden_dim,
            batch_first=True,
        )

        # --------------------------------------------------
        # Tête LogNormal AFT
        # --------------------------------------------------

        self.aft_head = nn.Sequential(

            nn.Linear(
                hidden_dim + static_dim,
                hidden_dim,
            ),

            nn.ReLU(),

            nn.Dropout(dropout),

            nn.Linear(
                hidden_dim,
                2,
            ),

        )

    def forward(
        self,
        sequence,
        lengths,
        static,
    ):
        """
        Parameters
        ----------
        sequence : Tensor
            Shape : (batch_size, seq_len, input_dim)

        lengths : Tensor
            Longueur réelle de chaque séquence.

        static : Tensor
            Variables statiques.

        Returns
        -------
        dict
            μ, σ et log(σ) de la distribution LogNormale.
        """

        # --------------------------------------------------
        # Ignore les observations de padding
        # --------------------------------------------------

        packed_sequence = pack_padded_sequence(
            sequence,
            lengths.cpu(),
            batch_first=True,
            enforce_sorted=False,
        )

        # --------------------------------------------------
        # Encodage de la séquence longitudinale
        # --------------------------------------------------

        _, hidden = self.gru(packed_sequence)

        # --------------------------------------------------
        # Dernier état caché du GRU
        # --------------------------------------------------

        h_final = hidden[-1]

        # --------------------------------------------------
        # Concaténation avec les variables statiques
        # --------------------------------------------------

        x = torch.cat(
            [h_final, static],
            dim=1,
        )

        # --------------------------------------------------
        # Paramètres de la distribution LogNormale
        # --------------------------------------------------

        aft_params = self.aft_head(x)

        mu = aft_params[:, 0]

        log_sigma = aft_params[:, 1]

        sigma = torch.exp(log_sigma)

        return {
            "mu": mu,
            "sigma": sigma,
            "log_sigma": log_sigma,
        }

In [ ]:
# ==========================================================
# FONCTION DE PERTE LOGNORMAL AFT
# ==========================================================
#
# La fonction de perte correspond au négatif de la
# log-vraisemblance d'un modèle Accelerated Failure Time
# (AFT) avec une distribution LogNormale.
#
# Contribution de chaque patient :
#
#   - Événement observé :
#       log(f(t))
#
#   - Patient censuré :
#       log(S(t))
#
# où :
#   f(t) est la densité de probabilité,
#   S(t) = 1 - F(t) est la fonction de survie.
#
# La perte finale est la moyenne du négatif de la
# log-vraisemblance sur le mini-batch.
# ==========================================================

def aft_loss(
    mu,
    sigma,
    time_to_event,
    event,
):
    """
    Calcule la perte LogNormal AFT.

    Parameters
    ----------
    mu : Tensor
        Paramètre de localisation.

    sigma : Tensor
        Paramètre d'échelle (> 0).

    time_to_event : Tensor
        Temps jusqu'à l'événement ou la censure.

    event : Tensor
        Indicateur d'événement
        (1 = événement, 0 = censuré).

    Returns
    -------
    Tensor
        Négatif de la log-vraisemblance moyenne.
    """

    # --------------------------------------------------
    # Distribution LogNormale
    # --------------------------------------------------

    dist = LogNormal(
        loc=mu,
        scale=sigma,
    )

    # --------------------------------------------------
    # Contribution des patients avec événement
    # --------------------------------------------------

    log_pdf = dist.log_prob(time_to_event)

    # --------------------------------------------------
    # Contribution des patients censurés
    # --------------------------------------------------

    survival = 1.0 - dist.cdf(time_to_event)

    log_survival = torch.log(
        survival.clamp(min=1e-8)
    )

    # --------------------------------------------------
    # Log-vraisemblance individuelle
    # --------------------------------------------------

    log_likelihood = (
        event * log_pdf
        + (1.0 - event) * log_survival
    )

    # --------------------------------------------------
    # Négatif de la log-vraisemblance moyenne
    # --------------------------------------------------

    return -log_likelihood.mean()

In [ ]:
# ==========================================================
# INITIALISATION DU MODÈLE ET DE L'OPTIMISEUR
# ==========================================================
#
# Le modèle GRU-AFT est entraîné à l'aide de l'optimiseur
# Adam en minimisant la fonction de perte LogNormal AFT.
#
# La graine aléatoire est fixée afin de garantir la
# reproductibilité des résultats.
# ==========================================================
SEED = 42

torch.manual_seed(SEED)

model = GRUAFT()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3,
)

# ==========================================================
# ÉTAPE D'ENTRAÎNEMENT
# ==========================================================
#
# Cette fonction réalise une itération d'entraînement sur un
# mini-batch :
#
#   1. remise à zéro des gradients ;
#   2. propagation avant (forward) ;
#   3. calcul de la perte LogNormal AFT ;
#   4. rétropropagation ;
#   5. mise à jour des paramètres.
# ==========================================================

# ==========================================================
# ÉTAPE D'ENTRAÎNEMENT
# ==========================================================

def train_step(
    model,
    batch,
    optimizer,
):
    """
    Réalise une étape d'entraînement sur un mini-batch.
    """

    optimizer.zero_grad()

    # --------------------------------------------------
    # Transfert du mini-batch vers le GPU
    # --------------------------------------------------

    sequence = batch["sequence"].to(device)

    # pack_padded_sequence attend les longueurs sur le CPU
    lengths = batch["lengths"]

    static = batch["static"].to(device)

    time_to_event = batch["time_to_event"].to(device)

    event = batch["event"].to(device)

    # --------------------------------------------------
    # Propagation avant
    # --------------------------------------------------

    outputs = model(
        sequence,
        lengths,
        static,
    )

    # --------------------------------------------------
    # Calcul de la perte
    # --------------------------------------------------

    loss = aft_loss(
        outputs["mu"],
        outputs["sigma"],
        time_to_event,
        event,
    )

    # --------------------------------------------------
    # Rétropropagation
    # --------------------------------------------------

    loss.backward()

    optimizer.step()

    return loss.item()

In [ ]:
# ==========================================================
# ENTRAÎNEMENT D'UNE ÉPOQUE
# ==========================================================
#
# Cette fonction réalise une époque complète d'entraînement.
# Chaque mini-batch est traité successivement et la perte
# moyenne sur l'ensemble de l'époque est retournée.
#
# Le DataLoader est parcouru une seule fois.
# ==========================================================

def train_epoch(
    model,
    train_loader,
    optimizer,
):
    """
    Réalise une époque complète d'entraînement.

    Parameters
    ----------
    model : nn.Module
        Modèle GRU-AFT.

    train_loader : DataLoader
        DataLoader du jeu d'entraînement.

    optimizer : torch.optim.Optimizer
        Optimiseur utilisé pour l'entraînement.

    Returns
    -------
    float
        Perte moyenne sur l'époque.
    """

    model.train()

    losses = []

    for batch in train_loader:

        loss = train_step(
            model,
            batch,
            optimizer,
        )

        losses.append(loss)

    return float(np.mean(losses))

In [ ]:
# ==========================================================
# DATASETS POUR LE RÉENTRAÎNEMENT FINAL
# ==========================================================
#
# Après la recherche des hyperparamètres par validation
# croisée sur le jeu d'entraînement, le modèle est
# réentraîné sur l'ensemble du jeu d'entraînement.
#
# Les performances sont ensuite évaluées sur :
#   - le jeu de validation interne ;
#   - le jeu de validation externe.
# ==========================================================

# ==========================================================
# DATASETS
# ==========================================================

final_train_dataset = DynamicSurvivalDataset(
    train_dynamic
)

internal_test_dataset = DynamicSurvivalDataset(
    internal_test_dynamic
)

external_test_dataset = DynamicSurvivalDataset(
    external_test_dynamic
)

# ==========================================================
# DATALOADERS
# ==========================================================

final_train_loader = DataLoader(

    final_train_dataset,

    batch_size=64,

    shuffle=True,

    collate_fn=survival_collate_fn,

)

internal_test_loader = DataLoader(

    internal_test_dataset,

    batch_size=64,

    shuffle=False,

    collate_fn=survival_collate_fn,

)

external_test_loader = DataLoader(

    external_test_dataset,

    batch_size=64,

    shuffle=False,

    collate_fn=survival_collate_fn,

)

In [ ]:
# ==========================================================
# ÉVALUATION DE LA FONCTION DE PERTE
# ==========================================================
#
# Cette fonction évalue la perte LogNormal AFT sur un jeu de
# données sans mettre à jour les paramètres du modèle.
#
# Le modèle est placé en mode évaluation afin de désactiver
# les couches spécifiques à l'entraînement (par exemple le
# Dropout) et le calcul des gradients est désactivé pour
# réduire le temps de calcul et la consommation mémoire.
# ==========================================================

def evaluate_loss(
    model,
    loader,
):
    """
    Calcule la perte moyenne sur un DataLoader.
    """

    model.eval()

    losses = []

    with torch.no_grad():

        for batch in loader:

            # ----------------------------------------------
            # Transfert du mini-batch vers le GPU
            # ----------------------------------------------

            sequence = batch["sequence"].to(device)

            # pack_padded_sequence attend les longueurs sur le CPU
            lengths = batch["lengths"]

            static = batch["static"].to(device)

            time_to_event = batch["time_to_event"].to(device)

            event = batch["event"].to(device)

            # ----------------------------------------------
            # Propagation avant
            # ----------------------------------------------

            outputs = model(
                sequence,
                lengths,
                static,
            )

            # ----------------------------------------------
            # Calcul de la perte
            # ----------------------------------------------

            loss = aft_loss(
                outputs["mu"],
                outputs["sigma"],
                time_to_event,
                event,
            )

            losses.append(loss.item())

    return float(np.mean(losses))

In [ ]:
# ==========================================================
# PRÉDICTION DES TEMPS DE SURVIE
# ==========================================================
#
# Cette fonction génère les prédictions du modèle sur un jeu
# de données et retourne un DataFrame contenant :
#
#   - l'identifiant du patient ;
#   - le temps observé ;
#   - le temps prédit ;
#   - l'indicateur d'événement ;
#   - le nombre de mesures longitudinales.
#
# Le temps prédit correspond à la médiane de la distribution
# LogNormale :
#
#     médiane = exp(μ)
#
# Ce choix fournit une estimation robuste du temps jusqu'à
# l'événement.
# ==========================================================

def predict_survival(
    model,
    loader,
):
    """
    Génère les prédictions de survie d'un modèle GRU-AFT.

    Parameters
    ----------
    model : nn.Module
        Modèle entraîné.

    loader : DataLoader
        Jeu de données à prédire.

    Returns
    -------
    pandas.DataFrame
        Tableau contenant les temps observés et prédits.
    """

    model.eval()

    patient_ids = []

    y_true = []
    y_pred = []

    events = []
    n_measures = []

    with torch.no_grad():

        for batch in loader:

            # --------------------------------------------------
            # Envoi des tenseurs sur le même device que le modèle
            # --------------------------------------------------

            sequence = batch["sequence"].to(device)

            lengths = batch["lengths"].to(device)

            static = batch["static"].to(device)

            outputs = model(

                sequence,

                lengths,

                static,

            )

            # --------------------------------------------------
            # Temps prédit
            #
            # Médiane de la distribution LogNormale :
            # median = exp(mu)
            # --------------------------------------------------

            predicted_time = torch.exp(outputs["mu"])

            # --------------------------------------------------
            # Identifiants des patients
            # --------------------------------------------------

            patient_ids.extend(batch["patient_id"])

            # --------------------------------------------------
            # Temps observés
            # --------------------------------------------------

            y_true.extend(

                batch["time_to_event"].cpu().numpy()

            )

            # --------------------------------------------------
            # Temps prédits
            # --------------------------------------------------

            y_pred.extend(

                predicted_time.cpu().numpy()

            )

            # --------------------------------------------------
            # Indicateurs d'événement
            # --------------------------------------------------

            events.extend(

                batch["event"].cpu().numpy()

            )

            # --------------------------------------------------
            # Nombre de mesures longitudinales
            # --------------------------------------------------

            n_measures.extend(

                lengths.cpu().numpy()

            )

    return pd.DataFrame(

        {

            "patient_id": patient_ids,

            "time_to_event": y_true,

            "predicted_time": y_pred,

            "event": events,

            "n_measures": n_measures,

        }

    )

In [ ]:
# ==========================================================
# OBJECTIF OPTUNA
# Validation croisée stratifiée par patient
# ==========================================================

from sklearn.model_selection import StratifiedGroupKFold

def objective(trial):

    # ======================================================
    # REPRODUCTIBILITÉ
    # ======================================================

    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(SEED)
        torch.cuda.manual_seed_all(SEED)

    # ======================================================
    # HYPERPARAMÈTRES
    # ======================================================

    hidden_dim = trial.suggest_categorical(
        "hidden_dim",
        [32, 64, 128, 256]
    )

    dropout = trial.suggest_float(
        "dropout",
        0.0,
        0.5
    )

    learning_rate = trial.suggest_float(
        "learning_rate",
        1e-4,
        1e-2,
        log=True
    )

    weight_decay = trial.suggest_float(
        "weight_decay",
        1e-7,
        1e-3,
        log=True
    )

    batch_size = trial.suggest_categorical(
        "batch_size",
        [32, 64, 128]
    )

    # ======================================================
    # VALIDATION CROISÉE
    # ======================================================

    cv = StratifiedGroupKFold(
        n_splits=5,
        shuffle=True,
        random_state=SEED
    )

    fold_losses = []

    X = np.arange(train_dynamic.height)

    y = train_dynamic["event_irt"].to_numpy()

    groups = train_dynamic["patient_id"].to_numpy()

    # ======================================================
    # BOUCLE SUR LES FOLDS
    # ======================================================

    for train_idx, val_idx in cv.split(X, y, groups):

        train_fold = train_dynamic[train_idx]
        val_fold = train_dynamic[val_idx]

        train_dataset = DynamicSurvivalDataset(train_fold)
        val_dataset = DynamicSurvivalDataset(val_fold)

        generator = torch.Generator()
        generator.manual_seed(SEED)

        train_loader = DataLoader(
            train_dataset,
            batch_size=batch_size,
            shuffle=True,
            collate_fn=survival_collate_fn,
            generator=generator
        )

        val_loader = DataLoader(
            val_dataset,
            batch_size=batch_size,
            shuffle=False,
            collate_fn=survival_collate_fn
        )

        model = GRUAFT(
            input_dim=5,
            static_dim=7,
            hidden_dim=hidden_dim,
            dropout=dropout
        ).to(device)

        optimizer = torch.optim.Adam(
            model.parameters(),
            lr=learning_rate,
            weight_decay=weight_decay
        )

        # ==================================================
        # EARLY STOPPING
        # ==================================================

        best_val_loss = np.inf
        patience = 5
        patience_counter = 0

        for epoch in range(30):

            train_epoch(
                model,
                train_loader,
                optimizer
            )

            val_loss = evaluate_loss(
                model,
                val_loader
            )

            if val_loss < best_val_loss:

                best_val_loss = val_loss
                patience_counter = 0

            else:

                patience_counter += 1

            if patience_counter >= patience:
                break

        fold_losses.append(best_val_loss)

    # ======================================================
    # OBJECTIF OPTUNA
    # ======================================================

    return float(np.mean(fold_losses))

In [ ]:
# ==========================================================
# OPTIMISATION DES HYPERPARAMÈTRES AVEC OPTUNA
# ==========================================================
#
# Recherche des meilleurs hyperparamètres du modèle GRU-AFT
# par validation croisée stratifiée à 5 folds.
#
# L'objectif est de minimiser la perte AFT moyenne obtenue
# sur les folds de validation.
# ==========================================================

sampler = optuna.samplers.TPESampler(
    seed=SEED
)

study = optuna.create_study(
    direction="minimize",
    sampler=sampler
)

study.optimize(
    objective,
    n_trials=50
)

In [ ]:
# ==========================================================
# MEILLEURS HYPERPARAMÈTRES
# ==========================================================
#
# Récupère les hyperparamètres ayant minimisé la perte
# moyenne de validation obtenue lors de la validation
# croisée.
# ==========================================================

best_params = study.best_params

print("Best hyperparameters:")
print(best_params)

print("\nBest mean validation loss:")
print(study.best_value)

In [ ]:
# ==========================================================
# SAUVEGARDE DES MEILLEURS HYPERPARAMÈTRES
# ==========================================================

import json
import os

output_dir = "experiments/gru_final"
os.makedirs(output_dir, exist_ok=True)

with open(
    os.path.join(output_dir, "hyperparameters.json"),
    "w",
) as f:

    json.dump(
        best_params,
        f,
        indent=4,
    )

In [ ]:
# ==========================================================
# MODÈLE FINAL
# ==========================================================
#
# Création du modèle final à partir des meilleurs
# hyperparamètres trouvés par Optuna.
#
# Ce modèle sera ensuite entraîné sur l'ensemble du jeu
# train_dynamic avant d'être évalué sur les jeux interne
# et externe.
# ==========================================================

torch.manual_seed(SEED)

model = GRUAFT(

    input_dim=5,
    static_dim=7,

    hidden_dim=best_params["hidden_dim"],

    dropout=best_params["dropout"]

).to(device)

optimizer = torch.optim.Adam(

    model.parameters(),

    lr=best_params["learning_rate"],

    weight_decay=best_params["weight_decay"]

)

In [ ]:
# ==========================================================
# ENTRAÎNEMENT DU MODÈLE FINAL
# ==========================================================

from pathlib import Path

Path("experiments/gru_final").mkdir(
    parents=True,
    exist_ok=True
)

# --------------------------------------------------
# DataLoader du jeu d'entraînement complet
# --------------------------------------------------

final_train_dataset = DynamicSurvivalDataset(train_dynamic)

final_train_loader = DataLoader(
    final_train_dataset,
    batch_size=best_params["batch_size"],
    shuffle=True,
    collate_fn=survival_collate_fn,
)

# --------------------------------------------------
# Entraînement
# --------------------------------------------------

n_epochs = 30

train_losses = []

for epoch in range(n_epochs):

    train_loss = train_epoch(
        model,
        final_train_loader,
        optimizer,
    )

    train_losses.append(train_loss)

    print(
        f"Epoch {epoch+1:03d} | "
        f"Train loss = {train_loss:.4f}"
    )

# --------------------------------------------------
# Sauvegarde du modèle final
# --------------------------------------------------

torch.save(
    model.state_dict(),
    "experiments/gru_final/model.pt",
)

print()
print("=" * 60)
print("ENTRAÎNEMENT TERMINÉ")
print("=" * 60)

In [ ]:
# ==========================================================
# CHARGEMENT DU MODÈLE FINAL
# ==========================================================
#
# Recharge le modèle GRU-AFT entraîné afin de réaliser
# les prédictions sur les jeux de test interne et externe.
# ==========================================================

torch.manual_seed(SEED)

model_gru = GRUAFT(

    input_dim=5,
    static_dim=7,

    hidden_dim=best_params["hidden_dim"],

    dropout=best_params["dropout"]

).to(device)

model_gru.load_state_dict(

    torch.load(
        "experiments/gru_final/model.pt",
        map_location=device,
    )

)

model_gru.eval()

In [ ]:
# ==========================================================
# PRÉDICTIONS - JEU DE TEST INTERNE
# ==========================================================

pred_gru_internal = predict_survival(
    model_gru,
    internal_test_loader,
)

pred_gru_internal.head(25)

In [ ]:
# ==========================================================
# ÉVALUATION - HARRELL C-INDEX
# ==========================================================
#
# Évalue la capacité du modèle à classer correctement les
# patients selon leur risque d'atteindre l'IRT.
#
# Plus le C-index est proche de 1, meilleure est la
# discrimination du modèle.
# ==========================================================

c_index_internal = concordance_index(

    pred_gru_internal["time_to_event"],

    pred_gru_internal["predicted_time"],

    pred_gru_internal["event"]

)

print(f"Internal Harrell C-index : {c_index_internal:.4f}")

In [ ]:
# ==========================================================
# APERÇU DES PRÉDICTIONS
# ==========================================================

print(pred_gru_internal.head())

print()

print("Shape :", pred_gru_internal.shape)

print()

print(pred_gru_internal["event"].value_counts())

In [ ]:
# ==========================================================
# MAE ET RMSE PAR NOMBRE DE MESURES
# ==========================================================
#
# Les erreurs sont calculées uniquement chez les patients
# ayant présenté un événement (IRT).
#
# Les performances sont ensuite résumées pour chaque nombre
# de mesures longitudinales disponibles.
# ==========================================================

events_only = pred_gru_internal[
    pred_gru_internal["event"] == 1
]

results = []

for n_measures in sorted(events_only["n_measures"].unique()):

    subset = events_only[
        events_only["n_measures"] == n_measures
    ]

    if len(subset) < 5:
        continue

    mae = mean_absolute_error(

        subset["time_to_event"],

        subset["predicted_time"]

    )

    rmse = np.sqrt(

        mean_squared_error(

            subset["time_to_event"],

            subset["predicted_time"]

        )

    )

    results.append({

        "n_measures": n_measures,

        "n_events": len(subset),

        "mae": mae,

        "rmse": rmse,

    })

In [ ]:
# ==========================================================
# TABLEAU RÉCAPITULATIF DES PERFORMANCES
# ==========================================================

results_gru_internal = (
    pd.DataFrame(results)
    .sort_values("n_measures")
    .reset_index(drop=True)
)

results_gru_internal

In [ ]:
# ==========================================================
# PATIENT AVEC ÉVÉNEMENT AYANT LE PLUS DE VISITES
# ==========================================================

event_patient_counts = (

    pred_gru_internal
    .loc[pred_gru_internal["event"] == 1]
    .groupby("patient_id")
    .size()
    .sort_values(ascending=False)

)

patient_id = event_patient_counts.index[0]

patient_predictions = (

    pred_gru_internal
    .loc[pred_gru_internal["patient_id"] == patient_id]
    .sort_values("n_measures")

)

patient_predictions

In [ ]:
# ==========================================================
# ANALYSE DES ERREURS DE PRÉDICTION
# ==========================================================

events_only = (
    pred_gru_internal
    .loc[pred_gru_internal["event"] == 1]
    .copy()
)

events_only["error"] = (

    events_only["predicted_time"]

    -

    events_only["time_to_event"]

)

In [ ]:
# ==========================================================
# PLUS GROSSES SOUS-ESTIMATIONS
# ==========================================================

underestimations = (

    events_only
    .sort_values("error")
    .head(20)

)

underestimations

In [ ]:
# ==========================================================
# PLUS GROSSES SURESTIMATIONS
# ==========================================================

overestimations = (

    events_only
    .sort_values("error", ascending=False)
    .head(20)

)

overestimations

In [ ]:
# ==========================================================
# ÉVALUATION PAR NOMBRE DE MESURES LONGITUDINALES
# ==========================================================
#
# Les performances du modèle sont évaluées pour chaque
# nombre de mesures longitudinales disponibles
# (n_measures).
#
# Les métriques rapportées sont :
#   - Harrell C-index (tous les patients)
#   - MAE (patients avec événement uniquement)
#   - RMSE (patients avec événement uniquement)
# ==========================================================

results = []

for n in sorted(pred_gru_internal["n_measures"].unique()):

    subset = pred_gru_internal[
        pred_gru_internal["n_measures"] == n
    ]

    if len(subset) < 20:
        continue

    c_index = concordance_index(
        subset["time_to_event"],
        subset["predicted_time"],
        subset["event"]
    )

    if subset["event"].sum() >= 5:

        mae = mean_absolute_error(
            subset.loc[subset["event"] == 1, "time_to_event"],
            subset.loc[subset["event"] == 1, "predicted_time"]
        )

        rmse = np.sqrt(
            mean_squared_error(
                subset.loc[subset["event"] == 1, "time_to_event"],
                subset.loc[subset["event"] == 1, "predicted_time"]
            )
        )

    else:

        mae = np.nan
        rmse = np.nan

    results.append({

        "n_measures": n,

        "n_patients": len(subset),

        "n_events": int(subset["event"].sum()),

        "c_index": c_index,

        "mae": mae,

        "rmse": rmse,

    })

results_gru_internal = (
    pd.DataFrame(results)
    .round({
        "c_index": 3,
        "mae": 2,
        "rmse": 2,
    })
    .sort_values("n_measures")
    .reset_index(drop=True)
)

results_gru_internal

In [ ]:
# ==========================================================
# TEMPS OBSERVÉ VS TEMPS PRÉDIT
# ==========================================================
#
# Comparaison entre le temps restant observé avant l'IRT
# et le temps prédit par le modèle, pour les patients
# ayant présenté un événement.
# ==========================================================

plt.figure(figsize=(7, 7))

plt.scatter(
    events_only["time_to_event"],
    events_only["predicted_time"],
    alpha=0.5,
)

max_time = max(
    events_only["time_to_event"].max(),
    events_only["predicted_time"].max(),
)

plt.plot(
    [0, max_time],
    [0, max_time],
    "r--",
    linewidth=2,
    label="Prédiction parfaite",
)

plt.xlabel("Temps observé avant l'IRT (années)")
plt.ylabel("Temps prédit avant l'IRT (années)")
plt.title("Temps observé vs temps prédit")
plt.legend()

plt.tight_layout()
plt.show()

### ODE-RNN

In [ ]:
# ==========================================================
# CONSTRUCTION DES SÉQUENCES POUR LE MODÈLE ODE-RNN
# ==========================================================
#
# Les variables longitudinales sont regroupées sous forme
# de séquences chronologiques qui serviront d'entrée au
# modèle ODE-RNN.
#
# Chaque visite est représentée par :
#   - âge
#   - DFG (CKD-EPI)
#   - hypertension
#   - événement urologique
#
# Les temps des visites (time_history) sont conservés dans
# une colonne distincte afin d'être utilisés par le solveur
# ODE lors de la propagation entre deux observations.
# ==========================================================

def build_sequence_features_ode(row):
    """
    Construit la séquence d'observations longitudinales
    utilisée par le modèle ODE-RNN.

    Chaque élément de la séquence correspond à une visite
    et contient les variables cliniques observées.
    """

    return [

        [
            age,
            egfr,
            hypertension,
            urologic_event,
        ]

        for age, egfr, hypertension, urologic_event in zip(

            row["age_history"],
            row["egfr_history"],
            row["hypertension_history"],
            row["urologic_history"],

        )

    ]


def add_sequence_features_ode(df):
    """
    Ajoute la colonne 'sequence_features_ode' contenant
    les séquences longitudinales utilisées par l'ODE-RNN.
    """

    return df.with_columns(

        pl.struct([
            "age_history",
            "egfr_history",
            "hypertension_history",
            "urologic_history",
        ])
        .map_elements(
            build_sequence_features_ode,
            return_dtype=pl.List(
                pl.List(pl.Float64)
            )
        )
        .alias("sequence_features_ode")

    )


# ==========================================================
# AJOUT DES SÉQUENCES AUX JEUX DE DONNÉES
# ==========================================================

train_dynamic_ode = add_sequence_features_ode(
    train_dynamic
)

internal_test_dynamic_ode = add_sequence_features_ode(
    internal_test_dynamic
)

external_test_dynamic_ode = add_sequence_features_ode(
    external_test_dynamic
)

In [ ]:
# ==========================================================
# DATASET DYNAMIQUE DE SURVIE POUR LE MODÈLE ODE-RNN
# ==========================================================
#
# Ce Dataset PyTorch retourne un snapshot longitudinal
# correspondant à un patient à une visite donnée.
#
# Chaque échantillon contient :
#   - la séquence des observations longitudinales ;
#   - les temps associés aux observations ;
#   - les variables statiques ;
#   - le temps avant événement ou censure ;
#   - l'indicateur d'événement ;
#   - la longueur réelle de la séquence ;
#   - l'identifiant du patient.
# ==========================================================

class ODEDynamicSurvivalDataset(Dataset):

    def __init__(self, df):
        self.df = df

    # --------------------------------------------------
    # Nombre total de snapshots
    # --------------------------------------------------

    def __len__(self):
        return self.df.height

    # --------------------------------------------------
    # Retourne un snapshot
    # --------------------------------------------------

    def __getitem__(self, idx):

        row = self.df.row(idx, named=True)

        # --------------------------------------------------
        # Séquence longitudinale
        # Shape : (sequence_length, 4)
        # Variables :
        #   - âge
        #   - DFG
        #   - hypertension
        #   - événement urologique
        # --------------------------------------------------

        sequence = torch.tensor(
            row["sequence_features_ode"],
            dtype=torch.float32,
        )

        # --------------------------------------------------
        # Temps associés aux observations
        # Shape : (sequence_length,)
        #
        # Utilisés par le solveur ODE pour calculer
        # l'évolution de l'état latent entre deux visites.
        # --------------------------------------------------

        time_history = torch.tensor(
            row["time_history"],
            dtype=torch.float32,
        )

        # --------------------------------------------------
        # Variables statiques
        # Shape : (7,)
        # --------------------------------------------------

        static = torch.tensor(
            row["static_features"],
            dtype=torch.float32,
        )

        # --------------------------------------------------
        # Temps restant avant l'événement ou la censure
        # --------------------------------------------------

        time_to_event = torch.tensor(
            row["time_to_event"],
            dtype=torch.float32,
        )

        # --------------------------------------------------
        # Indicateur d'événement
        #
        # 1 : IRT observée
        # 0 : patient censuré
        # --------------------------------------------------

        event = torch.tensor(
            row["event_irt"],
            dtype=torch.float32,
        )

        # --------------------------------------------------
        # Longueur réelle de la séquence
        # --------------------------------------------------

        length = torch.tensor(
            len(row["sequence_features_ode"]),
            dtype=torch.long,
        )

        return {

            "sequence": sequence,

            "time_history": time_history,

            "static": static,

            "time_to_event": time_to_event,

            "event": event,

            "length": length,

            "patient_id": row["patient_id"],

        }

In [ ]:
# ==========================================================
# COLLATE FUNCTION POUR LE MODÈLE ODE-RNN
# ==========================================================
#
# Cette fonction regroupe les snapshots d'un mini-batch.
#
# Les séquences longitudinales et leurs temps associés
# sont paddés afin d'obtenir des tenseurs de taille
# identique.
#
# Les longueurs réelles sont conservées pour permettre
# au modèle de distinguer les observations réelles des
# valeurs de padding.
# ==========================================================

def ode_survival_collate_fn(batch):

    # --------------------------------------------------
    # Séquences longitudinales
    # Shape : (batch_size, max_seq_len, 4)
    # --------------------------------------------------

    sequences = [
        sample["sequence"]
        for sample in batch
    ]

    # --------------------------------------------------
    # Temps des observations
    # Shape : (batch_size, max_seq_len)
    # --------------------------------------------------

    time_histories = [
        sample["time_history"]
        for sample in batch
    ]

    # --------------------------------------------------
    # Padding des séquences
    # --------------------------------------------------

    padded_sequences = pad_sequence(
        sequences,
        batch_first=True,
        padding_value=0.0,
    )

    # --------------------------------------------------
    # Padding des temps d'observation
    # --------------------------------------------------

    padded_time_history = pad_sequence(
        time_histories,
        batch_first=True,
        padding_value=0.0,
    )

    # --------------------------------------------------
    # Longueurs réelles des séquences
    # --------------------------------------------------

    lengths = torch.stack([
        sample["length"]
        for sample in batch
    ])

    # --------------------------------------------------
    # Variables statiques
    # Shape : (batch_size, static_dim)
    # --------------------------------------------------

    static = torch.stack([
        sample["static"]
        for sample in batch
    ])

    # --------------------------------------------------
    # Temps avant événement ou censure
    # --------------------------------------------------

    time_to_event = torch.stack([
        sample["time_to_event"]
        for sample in batch
    ])

    # --------------------------------------------------
    # Indicateur d'événement
    # --------------------------------------------------

    event = torch.stack([
        sample["event"]
        for sample in batch
    ])

    # --------------------------------------------------
    # Identifiants patients
    #
    # Conservés uniquement pour l'analyse des résultats.
    # Ils ne sont jamais utilisés pendant l'entraînement.
    # --------------------------------------------------

    patient_ids = [
        sample["patient_id"]
        for sample in batch
    ]

    return {

        "sequence": padded_sequences,

        "time_history": padded_time_history,

        "lengths": lengths,

        "static": static,

        "time_to_event": time_to_event,

        "event": event,

        "patient_id": patient_ids,

    }

In [ ]:
# ==========================================================
# FONCTION ODE
# ==========================================================
#
# Cette classe paramètre le champ de vecteurs
# dh/dt = f(h) décrivant l'évolution continue de
# l'état latent entre deux observations.
#
# La dynamique est modélisée par un réseau de neurones
# entièrement connecté.
# ==========================================================

class ODEFunc(nn.Module):

    def __init__(
        self,
        hidden_dim,
        ode_hidden_dim=64,
    ):
        super().__init__()

        self.net = nn.Sequential(

            nn.Linear(
                hidden_dim,
                ode_hidden_dim,
            ),

            nn.Tanh(),

            nn.Linear(
                ode_hidden_dim,
                hidden_dim,
            ),

        )

    def forward(
        self,
        t,
        h,
    ):
        """
        Calcule la dérivée de l'état latent.

        Parameters
        ----------
        t : torch.Tensor
            Instant courant de l'intégration.
            Il est conservé pour être compatible avec les
            solveurs d'ODE, même si le champ de vecteurs
            ne dépend pas explicitement du temps.

        h : torch.Tensor
            État latent courant.

        Returns
        -------
        torch.Tensor
            Dérivée de l'état latent (dh/dt).
        """

        return self.net(h)

In [ ]:
# ==========================================================
# CLASSE DE BASE DES SOLVEURS ODE
# ==========================================================
#
# Cette classe définit l'interface commune des solveurs ODE
# utilisés pour faire évoluer l'état latent entre deux
# observations successives.
#
# Chaque solveur hérite de cette classe et implémente sa
# propre méthode d'intégration dans la fonction forward().
# ==========================================================

class ODESolver(nn.Module):

    def __init__(self, ode_func):
        """
        Parameters
        ----------
        ode_func : nn.Module
            Fonction représentant le champ de vecteurs
            dh/dt = f(h).
        """

        super().__init__()

        self.ode_func = ode_func

    def forward(
        self,
        h,
        delta_t,
        t=None,
    ):
        """
        Fait évoluer l'état latent sur un intervalle de temps.

        Parameters
        ----------
        h : torch.Tensor
            État latent courant.

        delta_t : torch.Tensor or float
            Temps écoulé entre deux observations.

        t : torch.Tensor, optional
            Temps absolu de départ. Conservé pour assurer
            la compatibilité avec certains solveurs.

        Returns
        -------
        torch.Tensor
            Nouvel état latent après intégration.
        """

        raise NotImplementedError(
            "Chaque solveur ODE doit implémenter sa propre méthode forward()."
        )

In [ ]:
# ==========================================================
# SOLVEUR D'EULER EXPLICITE
# ==========================================================
#
# Ce solveur approxime l'évolution de l'état latent entre
# deux observations successives à l'aide du schéma d'Euler :
#
#     h(t + Δt) = h(t) + Δt · f(h(t))
#
# Il s'agit de la méthode d'intégration numérique la plus
# simple. Elle est rapide mais moins précise que des
# méthodes d'ordre supérieur comme RK4 ou Dormand-Prince.
# ==========================================================

class EulerSolver(ODESolver):

    def forward(
        self,
        h,
        delta_t,
        t=None,
    ):
        """
        Fait évoluer l'état latent pendant un intervalle Δt
        à l'aide du schéma d'Euler explicite.

        Parameters
        ----------
        h : torch.Tensor
            État latent courant.

        delta_t : torch.Tensor
            Temps écoulé entre deux observations.

        t : torch.Tensor, optional
            Temps absolu courant. Si aucun temps n'est fourni,
            un tenseur de zéros est utilisé.

        Returns
        -------
        torch.Tensor
            Nouvel état latent après intégration.
        """

        # --------------------------------------------------
        # Temps initial
        # --------------------------------------------------

        if t is None:
            t = torch.zeros_like(delta_t)

        # --------------------------------------------------
        # Calcul de la dérivée dh/dt
        # --------------------------------------------------

        dh = self.ode_func(
            t,
            h,
        )

        # --------------------------------------------------
        # Intégration par Euler explicite
        # --------------------------------------------------

        h_next = h + delta_t.unsqueeze(1) * dh

        return h_next

In [ ]:
# ==========================================================
# SOLVEUR DE RUNGE-KUTTA D'ORDRE 4 (RK4)
# ==========================================================
#
# Ce solveur approxime l'évolution de l'état latent entre
# deux observations à l'aide de la méthode de
# Runge-Kutta d'ordre 4.
#
# Contrairement au schéma d'Euler, RK4 estime la dérivée
# en quatre points intermédiaires, ce qui améliore
# significativement la précision de l'intégration.
# ==========================================================

class RK4Solver(ODESolver):

    def forward(
        self,
        h,
        delta_t,
        t=None,
    ):
        """
        Fait évoluer l'état latent à l'aide de la méthode
        de Runge-Kutta d'ordre 4.

        Parameters
        ----------
        h : torch.Tensor
            État latent courant.

        delta_t : torch.Tensor
            Temps écoulé entre deux observations.

        t : torch.Tensor, optional
            Temps absolu courant.

        Returns
        -------
        torch.Tensor
            Nouvel état latent après intégration.
        """

        # --------------------------------------------------
        # Temps initial
        # --------------------------------------------------

        if t is None:
            t = torch.zeros_like(delta_t)

        dt = delta_t.unsqueeze(1)

        # --------------------------------------------------
        # Évaluation des quatre pentes de RK4
        # --------------------------------------------------

        k1 = self.ode_func(
            t,
            h,
        )

        k2 = self.ode_func(
            t + delta_t / 2,
            h + dt * k1 / 2,
        )

        k3 = self.ode_func(
            t + delta_t / 2,
            h + dt * k2 / 2,
        )

        k4 = self.ode_func(
            t + delta_t,
            h + dt * k3,
        )

        # --------------------------------------------------
        # Mise à jour de l'état latent
        # --------------------------------------------------

        h_next = h + dt * (

            k1
            + 2 * k2
            + 2 * k3
            + k4

        ) / 6

        return h_next

In [ ]:
# ==========================================================
# SOLVEUR DU POINT MILIEU (RK2)
# ==========================================================
#
# Ce solveur approxime l'évolution de l'état latent à
# l'aide de la méthode du point milieu (Runge-Kutta d'ordre 2).
#
# La dérivée est d'abord évaluée au temps courant, puis
# une seconde fois au milieu de l'intervalle afin
# d'obtenir une approximation plus précise que le schéma
# d'Euler, tout en restant moins coûteuse que RK4.
# ==========================================================

class MidpointSolver(ODESolver):

    def forward(
        self,
        h,
        delta_t,
        t=None,
    ):
        """
        Fait évoluer l'état latent à l'aide de la méthode
        du point milieu (RK2).

        Parameters
        ----------
        h : torch.Tensor
            État latent courant.

        delta_t : torch.Tensor
            Temps écoulé entre deux observations.

        t : torch.Tensor, optional
            Temps absolu courant.

        Returns
        -------
        torch.Tensor
            Nouvel état latent après intégration.
        """

        # --------------------------------------------------
        # Temps initial
        # --------------------------------------------------

        if t is None:
            t = torch.zeros_like(delta_t)

        dt = delta_t.unsqueeze(1)

        # --------------------------------------------------
        # Première évaluation de la dérivée
        # --------------------------------------------------

        k1 = self.ode_func(
            t,
            h,
        )

        # --------------------------------------------------
        # Évaluation de la dérivée au milieu de l'intervalle
        # --------------------------------------------------

        k2 = self.ode_func(
            t + delta_t / 2,
            h + dt * k1 / 2,
        )

        # --------------------------------------------------
        # Mise à jour de l'état latent
        # --------------------------------------------------

        h_next = h + dt * k2

        return h_next

In [ ]:
# ==========================================================
# BLOC D'INTÉGRATION ODE
# ==========================================================
#
# Ce module regroupe :
#   - la fonction ODE représentant le champ de vecteurs ;
#   - le solveur numérique utilisé pour intégrer cette
#     dynamique entre deux observations successives.
#
# Le solveur est choisi lors de l'initialisation afin de
# pouvoir comparer facilement différentes méthodes
# d'intégration (Euler, RK2, RK4, ...).
# ==========================================================

class ODEBlock(nn.Module):

    def __init__(
        self,
        hidden_dim,
        ode_hidden_dim=64,
        solver="euler",
    ):
        """
        Parameters
        ----------
        hidden_dim : int
            Dimension de l'état latent.

        ode_hidden_dim : int, default=64
            Nombre de neurones de la couche cachée de la
            fonction ODE.

        solver : {"euler", "midpoint", "rk4"}
            Méthode d'intégration numérique utilisée.
        """

        super().__init__()

        # --------------------------------------------------
        # Fonction représentant le champ de vecteurs
        # --------------------------------------------------

        self.func = ODEFunc(
            hidden_dim,
            ode_hidden_dim,
        )

        # --------------------------------------------------
        # Sélection du solveur numérique
        # --------------------------------------------------

        if solver == "euler":

            self.solver = EulerSolver(self.func)

        elif solver == "midpoint":

            self.solver = MidpointSolver(self.func)

        elif solver == "rk4":

            self.solver = RK4Solver(self.func)

        else:

            raise ValueError(
                f"Unknown solver '{solver}'. "
                "Available solvers are: "
                "'euler', 'midpoint' and 'rk4'."
            )

    def forward(
        self,
        h,
        delta_t,
    ):
        """
        Fait évoluer l'état latent pendant un intervalle
        de temps Δt.

        Parameters
        ----------
        h : torch.Tensor
            État latent courant.

        delta_t : torch.Tensor
            Temps écoulé depuis la dernière observation.

        Returns
        -------
        torch.Tensor
            Nouvel état latent après intégration.
        """

        return self.solver(
            h,
            delta_t,
        )

In [ ]:
# ==========================================================
# ENCODEUR ODE-RNN
# ==========================================================
#
# Cet encodeur transforme une séquence longitudinale de
# mesures cliniques en une représentation latente unique.
#
# Pour chaque visite :
#   1. les variables observées sont encodées ;
#   2. le GRU met à jour l'état latent avec la nouvelle
#      information clinique ;
#   3. l'ODE fait évoluer cet état latent jusqu'à la
#      visite suivante en tenant compte du temps écoulé.
#
# Les patients d'un même batch pouvant avoir un nombre de
# visites différent, un masque est utilisé afin de ne
# mettre à jour que les séquences encore valides.
# ==========================================================

class ODERNNEncoder(nn.Module):

    def __init__(
        self,
        input_dim=4,
        hidden_dim=64,
        ode_hidden_dim=64,
        solver="euler",
    ):
        """
        Parameters
        ----------
        input_dim : int
            Nombre de variables longitudinales.

        hidden_dim : int
            Dimension de l'état latent.

        ode_hidden_dim : int
            Dimension cachée de la fonction ODE.

        solver : {"euler", "midpoint", "rk4"}
            Solveur numérique utilisé pour intégrer
            la dynamique continue.
        """

        super().__init__()

        self.hidden_dim = hidden_dim

        # --------------------------------------------------
        # Encode les variables observées à chaque visite
        # --------------------------------------------------

        self.input_encoder = nn.Sequential(

            nn.Linear(
                input_dim,
                hidden_dim,
            ),

            nn.ReLU(),

        )

        # --------------------------------------------------
        # Mise à jour discrète de l'état latent
        # --------------------------------------------------

        self.gru_cell = nn.GRUCell(

            input_size=hidden_dim,
            hidden_size=hidden_dim,

        )

        # --------------------------------------------------
        # Évolution continue entre deux visites
        # --------------------------------------------------

        self.ode_block = ODEBlock(

            hidden_dim=hidden_dim,
            ode_hidden_dim=ode_hidden_dim,
            solver=solver,

        )

    def forward(
        self,
        sequence,
        time_history,
        lengths,
    ):
        """
        Encode une séquence longitudinale.

        Parameters
        ----------
        sequence : torch.Tensor
            Variables longitudinales de forme
            (batch, n_visites, input_dim).

        time_history : torch.Tensor
            Âges ou temps des visites.

        lengths : torch.Tensor
            Nombre réel de visites de chaque patient.

        Returns
        -------
        torch.Tensor
            Représentation latente finale de chaque patient.
        """

        batch_size = sequence.size(0)
        max_len = sequence.size(1)
        device = sequence.device

        # --------------------------------------------------
        # Initialisation de l'état latent
        # --------------------------------------------------

        h = torch.zeros(
            batch_size,
            self.hidden_dim,
            device=device,
        )

        # --------------------------------------------------
        # Parcours chronologique des visites
        # --------------------------------------------------

        for t in range(max_len):

            # ----------------------------------------------
            # Patients possédant une visite à cet instant
            # ----------------------------------------------

            valid_obs = (t < lengths)

            # ----------------------------------------------
            # Encodage des variables observées
            # ----------------------------------------------

            x = self.input_encoder(
                sequence[:, t]
            )

            # ----------------------------------------------
            # Mise à jour du GRU
            # ----------------------------------------------

            h_candidate = self.gru_cell(
                x,
                h,
            )

            h = torch.where(
                valid_obs.unsqueeze(1),
                h_candidate,
                h,
            )

            # ----------------------------------------------
            # Propagation continue jusqu'à la visite suivante
            # ----------------------------------------------

            if t < max_len - 1:

                valid_transition = (t + 1 < lengths)

                delta_t = (
                    time_history[:, t + 1]
                    - time_history[:, t]
                )

                delta_t = torch.where(
                    valid_transition,
                    delta_t,
                    torch.zeros_like(delta_t),
                )

                h_candidate = self.ode_block(
                    h,
                    delta_t,
                )

                h = torch.where(
                    valid_transition.unsqueeze(1),
                    h_candidate,
                    h,
                )

        return h

In [ ]:
# ==========================================================
# TÊTE DE PRÉDICTION AFT
# ==========================================================
#
# Cette tête combine :
#   - la représentation dynamique produite par l'encodeur
#     ODE-RNN ;
#   - les variables statiques du patient.
#
# À partir de cette représentation commune, deux sorties
# sont apprises afin de paramétrer une distribution
# Log-Normale :
#   - μ : moyenne du logarithme du temps de survie ;
#   - σ : écart-type du logarithme du temps de survie.
# ==========================================================


class AFTHead(nn.Module):

    def __init__(
        self,
        hidden_dim,
        static_dim,
    ):
        """
        Parameters
        ----------
        hidden_dim : int
            Dimension de la représentation latente.

        static_dim : int
            Nombre de variables statiques.
        """

        super().__init__()

        # --------------------------------------------------
        # Représentation commune après fusion des variables
        # dynamiques et statiques
        # --------------------------------------------------

        self.shared = nn.Sequential(

            nn.Linear(
                hidden_dim + static_dim,
                hidden_dim,
            ),

            nn.ReLU(),

        )

        # --------------------------------------------------
        # Paramètre μ de la distribution Log-Normale
        # --------------------------------------------------

        self.mu_head = nn.Linear(
            hidden_dim,
            1,
        )

        # --------------------------------------------------
        # Paramètre σ de la distribution Log-Normale
        # --------------------------------------------------

        self.sigma_head = nn.Linear(
            hidden_dim,
            1,
        )

    def forward(
        self,
        hidden,
        static,
    ):
        """
        Prédit les paramètres de la distribution Log-Normale.

        Parameters
        ----------
        hidden : torch.Tensor
            Représentation dynamique produite par l'encodeur.

        static : torch.Tensor
            Variables statiques du patient.

        Returns
        -------
        tuple(torch.Tensor, torch.Tensor)
            μ et σ de la distribution Log-Normale.
        """

        # --------------------------------------------------
        # Fusion des informations dynamiques et statiques
        # --------------------------------------------------

        x = torch.cat(
            [
                hidden,
                static,
            ],
            dim=1,
        )

        x = self.shared(x)

        # --------------------------------------------------
        # Paramètre μ
        # --------------------------------------------------

        mu = self.mu_head(
            x,
        ).squeeze(1)

        # --------------------------------------------------
        # Paramètre σ
        # Softplus garantit une valeur strictement positive,
        # nécessaire pour une distribution Log-Normale.
        # --------------------------------------------------

        sigma = (
            F.softplus(
                self.sigma_head(x)
            ).squeeze(1)
            + 1e-6
        )

        return mu, sigma

In [ ]:
# ==========================================================
# MODÈLE ODE-RNN AVEC TÊTE LOG-NORMAL AFT
# ==========================================================
#
# Ce modèle combine :
#   1. un encodeur ODE-RNN qui résume l'historique
#      longitudinal du patient ;
#   2. une tête AFT qui fusionne cette représentation avec
#      les variables statiques afin de prédire les
#      paramètres de la distribution Log-Normale du temps
#      avant l'événement.
# ==========================================================

class ODERNNAFT(nn.Module):

    def __init__(
        self,
        input_dim=4,
        static_dim=7,
        hidden_dim=64,
        ode_hidden_dim=64,
        solver="euler",
    ):
        """
        Parameters
        ----------
        input_dim : int
            Nombre de variables longitudinales.

        static_dim : int
            Nombre de variables statiques.

        hidden_dim : int
            Dimension de l'état latent.

        ode_hidden_dim : int
            Dimension cachée de la fonction ODE.

        solver : {"euler", "midpoint", "rk4"}
            Solveur numérique utilisé pour intégrer
            l'évolution continue de l'état latent.
        """

        super().__init__()

        # --------------------------------------------------
        # Encodeur des trajectoires longitudinales
        # --------------------------------------------------

        self.encoder = ODERNNEncoder(

            input_dim=input_dim,
            hidden_dim=hidden_dim,
            ode_hidden_dim=ode_hidden_dim,
            solver=solver,

        )

        # --------------------------------------------------
        # Tête de prédiction AFT
        # --------------------------------------------------

        self.head = AFTHead(

            hidden_dim=hidden_dim,
            static_dim=static_dim,

        )

    def forward(
        self,
        sequence,
        time_history,
        lengths,
        static,
    ):
        """
        Prédit les paramètres de la distribution
        Log-Normale du temps avant l'événement.

        Parameters
        ----------
        sequence : torch.Tensor
            Variables longitudinales.

        time_history : torch.Tensor
            Temps ou âge des visites.

        lengths : torch.Tensor
            Nombre réel de visites par patient.

        static : torch.Tensor
            Variables statiques.

        Returns
        -------
        dict
            Dictionnaire contenant :
                - mu
                - sigma
        """

        # --------------------------------------------------
        # Encodage de la trajectoire longitudinale
        # --------------------------------------------------

        hidden = self.encoder(

            sequence=sequence,
            time_history=time_history,
            lengths=lengths,

        )

        # --------------------------------------------------
        # Paramètres de la distribution Log-Normale
        # --------------------------------------------------

        mu, sigma = self.head(

            hidden,
            static,

        )

        return {

            "mu": mu,
            "sigma": sigma,

        }

In [ ]:
# ==========================================================
# ÉVALUATION DE LA FONCTION DE PERTE
# ==========================================================
#
# Cette fonction calcule la perte moyenne du modèle sur un
# jeu de données (validation, test interne ou test externe)
# sans mettre à jour les poids du réseau.
# ==========================================================

def evaluate_loss_ode(
    model,
    loader,
):
    """
    Calcule la perte moyenne AFT sur un DataLoader.

    Parameters
    ----------
    model : nn.Module
        Modèle ODE-RNN AFT.

    loader : DataLoader
        Jeu de données à évaluer.

    Returns
    -------
    float
        Valeur moyenne de la perte AFT.
    """

    model.eval()

    losses = []

    with torch.no_grad():

        for batch in loader:

            # --------------------------------------------------
            # Données du batch
            # --------------------------------------------------

            sequence = batch["sequence"].to(device)
            time_history = batch["time_history"].to(device)
            lengths = batch["lengths"].to(device)
            static = batch["static"].to(device)

            time_to_event = batch["time_to_event"].to(device)
            event = batch["event"].to(device)

            # --------------------------------------------------
            # Prédictions
            # --------------------------------------------------

            outputs = model(

                sequence=sequence,
                time_history=time_history,
                lengths=lengths,
                static=static,

            )

            # --------------------------------------------------
            # Calcul de la perte AFT
            # --------------------------------------------------

            loss = aft_loss(

                outputs["mu"],
                outputs["sigma"],
                time_to_event,
                event,

            )

            losses.append(loss.item())

    return float(np.mean(losses))

In [ ]:
# ==========================================================
# PRÉPARATION DE LA VALIDATION CROISÉE
# ==========================================================
#
# La recherche d'hyperparamètres est réalisée à l'aide d'une
# validation croisée stratifiée par patient.
#
# Chaque patient est entièrement affecté à un seul fold afin
# d'éviter toute fuite d'information entre l'entraînement et
# la validation. La stratification est effectuée sur
# l'événement IRT afin de conserver une proportion similaire
# d'événements dans chaque fold.
# ==========================================================

from sklearn.model_selection import StratifiedGroupKFold

# --------------------------------------------------
# Un patient = un groupe
# --------------------------------------------------

patient_event = (

    train_dynamic_ode

    .group_by("patient_id")

    .agg(

        pl.max("event_irt").alias("event")

    )

)

# --------------------------------------------------
# Validation croisée stratifiée
# --------------------------------------------------

cv = StratifiedGroupKFold(

    n_splits=5,
    shuffle=True,
    random_state=SEED,

)

# --------------------------------------------------
# Informations utilisées par la validation croisée
# --------------------------------------------------

patient_ids = patient_event["patient_id"].to_numpy()

patient_events = patient_event["event"].to_numpy()

groups = patient_ids

print("=" * 60)
print("VALIDATION CROISÉE")
print("=" * 60)
print(f"Nombre de patients : {len(patient_ids)}")
print(f"Nombre de folds    : {cv.get_n_splits()}")
print(f"Patients avec IRT  : {patient_events.sum()}")
print(f"Patients censurés  : {len(patient_events) - patient_events.sum()}")

In [ ]:
# ==========================================================
# DATASET ET DATALOADER — TEST INTERNE
# ==========================================================

internal_test_dataset_ode = ODEDynamicSurvivalDataset(
    internal_test_dynamic_ode
)

internal_test_loader_ode = DataLoader(
    internal_test_dataset_ode,
    batch_size=64,
    shuffle=False,
    collate_fn=ode_survival_collate_fn,
)

# ==========================================================
# DATASET ET DATALOADER — TEST EXTERNE
# ==========================================================

external_test_dataset_ode = ODEDynamicSurvivalDataset(
    external_test_dynamic_ode
)

external_test_loader_ode = DataLoader(
    external_test_dataset_ode,
    batch_size=64,
    shuffle=False,
    collate_fn=ode_survival_collate_fn,
)

In [ ]:
# ==========================================================
# ÉTAPE D'ENTRAÎNEMENT
# ==========================================================
#
# Réalise une étape d'optimisation sur un batch :
#   1. passage en mode entraînement ;
#   2. propagation avant ;
#   3. calcul de la perte AFT ;
#   4. rétropropagation ;
#   5. mise à jour des poids.
# ==========================================================

def train_step_ode(
    model,
    batch,
    optimizer,
):
    """
    Effectue une étape d'entraînement.

    Parameters
    ----------
    model : nn.Module
        Modèle ODE-RNN AFT.

    batch : dict
        Batch fourni par le DataLoader.

    optimizer : torch.optim.Optimizer
        Optimiseur utilisé pour la mise à jour des poids.

    Returns
    -------
    float
        Valeur de la perte sur le batch.
    """

    model.train()

    optimizer.zero_grad()

    # --------------------------------------------------
    # Transfert des données sur le device
    # --------------------------------------------------

    sequence = batch["sequence"].to(device)
    time_history = batch["time_history"].to(device)
    lengths = batch["lengths"].to(device)
    static = batch["static"].to(device)

    time_to_event = batch["time_to_event"].to(device)
    event = batch["event"].to(device)

    # --------------------------------------------------
    # Propagation avant
    # --------------------------------------------------

    outputs = model(

        sequence=sequence,
        time_history=time_history,
        lengths=lengths,
        static=static,

    )

    # --------------------------------------------------
    # Calcul de la perte AFT
    # --------------------------------------------------

    loss = aft_loss(

        outputs["mu"],
        outputs["sigma"],
        time_to_event,
        event,

    )

    # --------------------------------------------------
    # Rétropropagation et mise à jour des poids
    # --------------------------------------------------

    loss.backward()

    optimizer.step()

    return loss.item()

In [ ]:
# ==========================================================
# ENTRAÎNEMENT D'UNE ÉPOQUE
# ==========================================================
#
# Cette fonction parcourt l'ensemble des batches du jeu
# d'entraînement, réalise une étape d'optimisation pour
# chacun d'eux et retourne la perte moyenne sur l'époque.
# ==========================================================

def train_epoch_ode(
    model,
    train_loader,
    optimizer,
):
    """
    Effectue une époque complète d'entraînement.

    Parameters
    ----------
    model : nn.Module
        Modèle ODE-RNN AFT.

    train_loader : DataLoader
        DataLoader du jeu d'entraînement.

    optimizer : torch.optim.Optimizer
        Optimiseur utilisé pour mettre à jour les poids.

    Returns
    -------
    float
        Perte moyenne sur l'époque.
    """

    model.train()

    epoch_losses = []

    for batch in train_loader:

        batch_loss = train_step_ode(

            model=model,
            batch=batch,
            optimizer=optimizer,

        )

        epoch_losses.append(batch_loss)

    return float(np.mean(epoch_losses))

In [ ]:
# ==========================================================
# PRÉDICTION DES TEMPS DE SURVIE
# ==========================================================
#
# Cette fonction applique le modèle à un jeu de données et
# retourne, pour chaque snapshot, le temps observé, le temps
# prédit ainsi que les informations nécessaires à
# l'évaluation des performances.
# ==========================================================

def predict_survival_ode(
    model,
    loader,
):
    """
    Prédit le temps restant avant l'événement.

    Parameters
    ----------
    model : nn.Module
        Modèle ODE-RNN AFT.

    loader : DataLoader
        Jeu de données à prédire.

    Returns
    -------
    pandas.DataFrame
        Tableau contenant les prédictions et les
        informations associées.
    """

    model.eval()

    patient_ids = []

    observed_times = []
    predicted_times = []

    events = []
    n_measures = []

    with torch.no_grad():

        for batch in loader:

            # --------------------------------------------------
            # Transfert des données sur le device
            # --------------------------------------------------

            sequence = batch["sequence"].to(device)
            time_history = batch["time_history"].to(device)
            lengths = batch["lengths"].to(device)
            static = batch["static"].to(device)

            # --------------------------------------------------
            # Prédictions du modèle
            # --------------------------------------------------

            outputs = model(

                sequence=sequence,
                time_history=time_history,
                lengths=lengths,
                static=static,

            )

            # --------------------------------------------------
            # Temps médian prédit
            #
            # Pour une loi Log-Normale paramétrée par μ et σ,
            # la médiane est égale à exp(μ).
            # --------------------------------------------------

            predicted_time = torch.exp(
                outputs["mu"]
            )

            # --------------------------------------------------
            # Sauvegarde des résultats du batch
            # --------------------------------------------------

            patient_ids.extend(batch["patient_id"])

            observed_times.extend(
                batch["time_to_event"].cpu().numpy()
            )

            predicted_times.extend(
                predicted_time.cpu().numpy()
            )

            events.extend(
                batch["event"].cpu().numpy()
            )

            n_measures.extend(
                batch["lengths"].cpu().numpy()
            )

    return pd.DataFrame(

        {

            "patient_id": patient_ids,
            "time_to_event": observed_times,
            "predicted_time": predicted_times,
            "event": events,
            "n_measures": n_measures,

        }

    )

In [ ]:
from lifelines.utils import concordance_index

def evaluate_cindex_ode(model, loader):

    model.eval()

    pred_times = []
    event_times = []
    events = []

    with torch.no_grad():

        for batch in loader:

            sequence = batch["sequence"].to(device)
            lengths = batch["lengths"].to(device)
            static = batch["static"].to(device)

            outputs = model(
                sequence,
                lengths,
                static
            )

            pred = torch.exp(outputs["mu"])

            pred_times.extend(
                pred.cpu().numpy().ravel()
            )

            event_times.extend(
                batch["time"].numpy()
            )

            events.extend(
                batch["event"].numpy()
            )

    return concordance_index(
        event_times,
        -np.asarray(pred_times),
        events
    )

In [ ]:
# ==========================================================
# OBJECTIF OPTUNA
# ==========================================================

import copy

def objective_ode(trial):

    # --------------------------------------------------
    # Reproductibilité
    # --------------------------------------------------

    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(SEED)
        torch.cuda.manual_seed_all(SEED)

    # --------------------------------------------------
    # Hyperparamètres
    # --------------------------------------------------

    hidden_dim = trial.suggest_categorical(
        "hidden_dim",
        [32, 64, 128, 256]
    )

    ode_hidden_dim = trial.suggest_categorical(
        "ode_hidden_dim",
        [32, 64, 128, 256]
    )

    learning_rate = trial.suggest_float(
        "learning_rate",
        1e-4,
        3e-3,
        log=True
    )

    batch_size = trial.suggest_categorical(
        "batch_size",
        [32, 64, 128]
    )

    solver = trial.suggest_categorical(
        "solver",
        [
            "euler",
            # "midpoint",
            "rk4"
        ]
    )

    # --------------------------------------------------
    # Validation croisée
    # --------------------------------------------------

    fold_scores = []

    for fold, (train_idx, val_idx) in enumerate(

        cv.split(
            X=patient_ids,
            y=patient_events,
            groups=groups
        ),

        start=1

    ):

        # ----------------------------------------------
        # Patients du fold
        # ----------------------------------------------

        train_ids = patient_ids[train_idx]
        val_ids = patient_ids[val_idx]

        # ----------------------------------------------
        # Données du fold
        # ----------------------------------------------

        train_fold = train_dynamic_ode.filter(
            pl.col("patient_id").is_in(train_ids)
        )

        val_fold = train_dynamic_ode.filter(
            pl.col("patient_id").is_in(val_ids)
        )

        # ----------------------------------------------
        # Datasets
        # ----------------------------------------------

        train_dataset = ODEDynamicSurvivalDataset(
            train_fold
        )

        val_dataset = ODEDynamicSurvivalDataset(
            val_fold
        )

        # ----------------------------------------------
        # DataLoaders
        # ----------------------------------------------

        train_loader = DataLoader(
            train_dataset,
            batch_size=batch_size,
            shuffle=True,
            collate_fn=ode_survival_collate_fn
        )

        val_loader = DataLoader(
            val_dataset,
            batch_size=batch_size,
            shuffle=False,
            collate_fn=ode_survival_collate_fn
        )

        # ----------------------------------------------
        # Modèle
        # ----------------------------------------------

        model = ODERNNAFT(

            input_dim=4,

            static_dim=7,

            hidden_dim=hidden_dim,

            ode_hidden_dim=ode_hidden_dim,

            solver=solver

        ).to(device)

        # ----------------------------------------------
        # Optimiseur
        # ----------------------------------------------

        optimizer = torch.optim.Adam(
            model.parameters(),
            lr=learning_rate
        )

        # ----------------------------------------------
        # Early stopping
        # ----------------------------------------------

        best_val_loss = float("inf")
        best_model = None

        patience = 5
        patience_counter = 0

        # ----------------------------------------------
        # Entraînement
        # ----------------------------------------------

        for epoch in range(30):

            train_epoch_ode(

                model=model,

                train_loader=train_loader,

                optimizer=optimizer

            )

            val_loss = evaluate_loss_ode(

                model=model,

                loader=val_loader

            )

            # ------------------------------------------
            # Early stopping
            # ------------------------------------------

            if val_loss < best_val_loss:

                best_val_loss = val_loss

                best_model = copy.deepcopy(
                    model.state_dict()
                )

                patience_counter = 0

            else:

                patience_counter += 1

            if patience_counter >= patience:

                break

        # ----------------------------------------------
        # Rechargement du meilleur modèle du fold
        # ----------------------------------------------

        model.load_state_dict(best_model)

        # ----------------------------------------------
        # C-index sur le fold
        # ----------------------------------------------

        c_index = evaluate_cindex_ode(
            model=model,
            loader=val_loader
        )

        fold_scores.append(c_index)

        # ----------------------------------------------
        # Optuna pruning
        # ----------------------------------------------

        mean_score = float(np.mean(fold_scores))

        trial.report(
            mean_score,
            fold
        )

        if trial.should_prune():

            raise optuna.TrialPruned()

    # --------------------------------------------------
    # Score final
    # --------------------------------------------------

    return float(np.mean(fold_scores))

In [ ]:
# ==========================================================
# OPTUNA STUDY
# ==========================================================

import optuna

g = torch.Generator()

g.manual_seed(42)

# --------------------------------------------------
# Sampler
# --------------------------------------------------

sampler_ode = optuna.samplers.TPESampler(
    seed=SEED
)

# --------------------------------------------------
# Study
# --------------------------------------------------

study_ode = optuna.create_study(
    study_name="ODE_RNN_AFT",
    direction="minimize",
    sampler=sampler_ode
)

# --------------------------------------------------
# Optimisation
# --------------------------------------------------

study_ode.optimize(
    objective_ode,
    n_trials=50,
    catch=(
        RuntimeError,
        ValueError,
    ),
)

In [ ]:
# ==========================================================
# MEILLEURS HYPERPARAMÈTRES
# ==========================================================

best_params_ode = study_ode.best_params

print("Meilleurs hyperparamètres :")

for key, value in best_params_ode.items():
    print(f"{key}: {value}")

print(f"\nMeilleure loss : {study_ode.best_value:.4f}")

In [ ]:
# ==========================================================
# DATASET FINAL
# ==========================================================

full_train_dataset_ode = ODEDynamicSurvivalDataset(
    train_dynamic_ode
)

full_train_loader_ode = DataLoader(

    full_train_dataset_ode,

    batch_size=best_params_ode["batch_size"],

    shuffle=True,

    collate_fn=ode_survival_collate_fn,

)

In [ ]:
# ==========================================================
# MODÈLE FINAL ODE-RNN
# ==========================================================

# --------------------------------------------------
# Reproductibilité
# --------------------------------------------------

random.seed(SEED)
np.random.seed(SEED)

torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

# --------------------------------------------------
# Modèle final
# --------------------------------------------------

final_model_ode = ODERNNAFT(

    input_dim=4,

    static_dim=7,

    hidden_dim=best_params_ode["hidden_dim"],

    ode_hidden_dim=best_params_ode["ode_hidden_dim"],

    solver=best_params_ode["solver"]

).to(device)

# --------------------------------------------------
# Optimiseur
# --------------------------------------------------

final_optimizer_ode = torch.optim.Adam(

    final_model_ode.parameters(),

    lr=best_params_ode["learning_rate"]

)

In [ ]:
# ==========================================================
# ENTRAÎNEMENT FINAL ODE-RNN
# ==========================================================

n_epochs = 100

patience = 8

best_train_loss = float("inf")

best_epoch = 0

epochs_without_improvement = 0

best_state_dict = None

# --------------------------------------------------
# Boucle d'entraînement
# --------------------------------------------------

for epoch in range(n_epochs):

    train_loss = train_epoch_ode(

        model=final_model_ode,

        train_loader=full_train_loader_ode,

        optimizer=final_optimizer_ode,

    )

    # --------------------------------------------------
    # Sauvegarde du meilleur modèle
    # --------------------------------------------------

    if train_loss < best_train_loss:

        best_train_loss = train_loss

        best_epoch = epoch + 1

        epochs_without_improvement = 0

        best_state_dict = {

            k: v.detach().cpu().clone()

            for k, v in final_model_ode.state_dict().items()

        }

        # torch.save(

        #     best_state_dict,

        #     experiment_path / "model.pt"

        # )

    else:

        epochs_without_improvement += 1

    # --------------------------------------------------
    # Affichage
    # --------------------------------------------------

    print(

        f"Epoch {epoch + 1:03d} | "

        f"Train Loss = {train_loss:.4f}"

    )

    # --------------------------------------------------
    # Early stopping
    # --------------------------------------------------

    if epochs_without_improvement >= patience:

        print()

        print(
            f"Early stopping à l'époque {epoch + 1}"
        )

        break

# --------------------------------------------------
# Recharge le meilleur modèle
# --------------------------------------------------

if best_state_dict is not None:

    final_model_ode.load_state_dict(

        best_state_dict

    )

# --------------------------------------------------
# Résumé
# --------------------------------------------------

print()
print("=" * 60)
print("ENTRAÎNEMENT TERMINÉ")
print("=" * 60)

print()

print(
    f"Meilleure époque      : {best_epoch}"
)

print(
    f"Meilleure train loss  : {best_train_loss:.4f}"
)

print()

print("Hyperparamètres retenus :")

print(
    f"  Solver          : {best_params_ode['solver']}"
)

print(
    f"  Hidden dim      : {best_params_ode['hidden_dim']}"
)

print(
    f"  ODE hidden dim  : {best_params_ode['ode_hidden_dim']}"
)

print(
    f"  Learning rate   : {best_params_ode['learning_rate']:.2e}"
)

print(
    f"  Batch size      : {best_params_ode['batch_size']}"
)

In [ ]:
# ==========================================================
# MEILLEURS HYPERPARAMÈTRES
# ==========================================================

best_params_ode = study_ode.best_params

print("Meilleurs hyperparamètres retenus :")

for key, value in best_params_ode.items():
    print(f"{key}: {value}")

print(f"\nMeilleure loss : {study_ode.best_value:.4f}")

#### E valuation du modèle

In [ ]:
# ==========================================================
# PRÉDICTIONS SUR LE JEU DE TEST INTERNE
# ==========================================================

predictions_internal_ode = predict_survival_ode(

    model=final_model_ode,

    loader=internal_test_loader_ode

)

# predictions_internal_ode.to_csv(
#     experiment_path / "predictions_internal.csv",
#     index=False
# )

In [ ]:
# afficher les premières prédictions

predictions_internal_ode.head(15)

In [ ]:
# ==========================================================
# ÉVALUATION GLOBALE
# ==========================================================

from lifelines.utils import concordance_index
from sklearn.metrics import mean_absolute_error, mean_squared_error
import json

# --------------------------------------------------
# Patients ayant présenté l'événement
# --------------------------------------------------

events_only = predictions_internal_ode[
    predictions_internal_ode["event"] == 1
].copy()

# --------------------------------------------------
# Métriques globales
# --------------------------------------------------

c_index = concordance_index(

    predictions_internal_ode["time_to_event"],

    predictions_internal_ode["predicted_time"],

    predictions_internal_ode["event"]

)

mae = mean_absolute_error(

    events_only["time_to_event"],

    events_only["predicted_time"]

)

rmse = np.sqrt(

    mean_squared_error(

        events_only["time_to_event"],

        events_only["predicted_time"]

    )

)

print("=" * 60)
print("ODE-RNN-AFT")
print("=" * 60)

print()

print(f"C-index : {c_index:.4f}")
print(f"MAE      : {mae:.3f} ans")
print(f"RMSE     : {rmse:.3f} ans")


In [ ]:
# ==========================================================
# PERFORMANCES SELON LE NOMBRE DE VISITES
# ==========================================================

results = []

for n in sorted(predictions_internal_ode["n_measures"].unique()):

    # --------------------------------------------------
    # Tous les patients du snapshot
    # --------------------------------------------------

    subset_all = predictions_internal_ode[
        predictions_internal_ode["n_measures"] == n
    ]

    # --------------------------------------------------
    # Patients ayant présenté l'événement
    # --------------------------------------------------

    subset_events = subset_all[
        subset_all["event"] == 1
    ]

    if len(subset_events) < 5:
        continue

    # --------------------------------------------------
    # C-index
    # --------------------------------------------------

    c_index = concordance_index(

        subset_all["time_to_event"],

        subset_all["predicted_time"],

        subset_all["event"]

    )

    # --------------------------------------------------
    # MAE
    # --------------------------------------------------

    mae = mean_absolute_error(

        subset_events["time_to_event"],

        subset_events["predicted_time"]

    )

    # --------------------------------------------------
    # RMSE
    # --------------------------------------------------

    rmse = np.sqrt(

        mean_squared_error(

            subset_events["time_to_event"],

            subset_events["predicted_time"]

        )

    )

    results.append({

        "n_measures": n,

        "n_patients": len(subset_all),

        "n_events": len(subset_events),

        "c_index": c_index,

        "mae": mae,

        "rmse": rmse

    })

results_ode = (

    pd.DataFrame(results)

    .sort_values("n_measures")

    .reset_index(drop=True)

)

display(results_ode.round(3))


In [ ]:
# ==========================================================
# MAE ET RMSE SELON LE NOMBRE DE VISITES
# ==========================================================

import plotly.graph_objects as go

fig = go.Figure()

# --------------------------------------------------
# MAE
# --------------------------------------------------

fig.add_trace(

    go.Scatter(

        x=results_ode["n_measures"],

        y=results_ode["mae"],

        mode="lines+markers+text",

        name="MAE",

        text=results_ode["n_events"],

        textposition="top center",

        line=dict(width=3),

        marker=dict(size=8),

        hovertemplate=(
            "Visites : %{x}<br>"
            "MAE : %{y:.2f} ans<br>"
            "Patients : %{text}<extra></extra>"
        )

    )

)

# --------------------------------------------------
# RMSE
# --------------------------------------------------

fig.add_trace(

    go.Scatter(

        x=results_ode["n_measures"],

        y=results_ode["rmse"],

        mode="lines+markers+text",

        name="RMSE",

        text=results_ode["n_events"],

        textposition="bottom center",

        line=dict(width=3),

        marker=dict(size=8),

        hovertemplate=(
            "Visites : %{x}<br>"
            "RMSE : %{y:.2f} ans<br>"
            "Patients : %{text}<extra></extra>"
        )

    )

)

# --------------------------------------------------
# Mise en forme
# --------------------------------------------------

fig.update_layout(

    title="Performance selon le nombre de visites",

    xaxis_title="Nombre de visites",

    yaxis_title="Erreur (années)",

    template="plotly_white",

    hovermode="x unified",

    width=850,

    height=500,

    legend=dict(

        orientation="h",

        y=1.08,

        x=0.5,

        xanchor="center"

    )

)

fig.show()

In [ ]:
# ==========================================================
# C-INDEX SELON LE NOMBRE DE VISITES
# ==========================================================

fig = go.Figure()

fig.add_trace(

    go.Scatter(

        x=results_ode["n_measures"],

        y=results_ode["c_index"],

        mode="lines+markers+text",

        text=results_ode["n_events"],

        textposition="top center",

        name="C-index",

        line=dict(width=3),

        marker=dict(size=8),

        hovertemplate=(

            "Visites : %{x}<br>"

            "C-index : %{y:.3f}<br>"

            "Événements : %{text}<extra></extra>"

        )

    )

)

fig.update_layout(

    title="C-index selon le nombre de visites",

    xaxis_title="Nombre de visites",

    yaxis_title="C-index",

    template="plotly_white",

    hovermode="x unified",

    width=850,

    height=500

)

fig.show()

In [ ]:
# ==========================================================
# PLUS GROSSES SUR-ESTIMATIONS
# ==========================================================

events_only = predictions_internal_ode[
    predictions_internal_ode["event"] == 1
].copy()

events_only["prediction_error"] = (

    events_only["predicted_time"]

    - events_only["time_to_event"]

)

over_predictions = (

    events_only[
        events_only["prediction_error"] > 0
    ]

    .sort_values(

        "prediction_error",

        ascending=False

    )

    .reset_index(drop=True)

)

display(

    over_predictions[[

        "patient_id",

        "n_measures",

        "time_to_event",

        "predicted_time",

        "prediction_error"

    ]].head(10)

)

In [ ]:
# ==========================================================
# PLUS GROSSES SOUS-ESTIMATIONS
# ==========================================================

under_predictions = (

    events_only[
        events_only["prediction_error"] < 0
    ]

    .sort_values(

        "prediction_error"

    )

    .reset_index(drop=True)

)

display(

    under_predictions[[

        "patient_id",

        "n_measures",

        "time_to_event",

        "predicted_time",

        "prediction_error"

    ]].head(10)

)

In [ ]:
# ==========================================================
# COMPARAISON GRU vs ODE-RNN
# Une ligne = un snapshot d'un patient
# ==========================================================

comparison = (

    predictions_internal_gru[
        [
            "patient_id",
            "event",
            "n_measures",
            "time_to_event",
            "predicted_time"
        ]
    ]

    .rename(
        columns={
            "predicted_time": "gru_prediction"
        }
    )

    .merge(

        predictions_internal_ode[
            [
                "patient_id",
                "n_measures",
                "predicted_time"
            ]
        ].rename(
            columns={
                "predicted_time": "ode_prediction"
            }
        ),

        on=[
            "patient_id",
            "n_measures"
        ],

        how="inner"

    )

    .merge(

        internal_test_dynamic

        .select(
            [
                "patient_id",
                "n_measures",
                "current_ckd_epi",
                "age_creat",
                "hypertension_at_t",
                "urologic_event_at_t"
            ]
        )

        .rename(
            {
                "current_ckd_epi": "egfr",
                "age_creat": "age",
                "hypertension_at_t": "hypertension",
                "urologic_event_at_t": "urologic_event"
            }
        )

        .to_pandas(),

        on=[
            "patient_id",
            "n_measures"
        ],

        how="left"

    )

)

# --------------------------------------------------
# Erreurs
# --------------------------------------------------

comparison["gru_error"] = (

    comparison["gru_prediction"]

    - comparison["time_to_event"]

)

comparison["ode_error"] = (

    comparison["ode_prediction"]

    - comparison["time_to_event"]

)

comparison["gru_abs_error"] = (

    comparison["gru_error"].abs()

)

comparison["ode_abs_error"] = (

    comparison["ode_error"].abs()

)

comparison.head()